In [1]:
pip install transformers datasets torch pandas scikit-learn accelerate

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path

main_dir = Path.cwd()
print(main_dir)

In [3]:
PATH_GAMETOX = "Existing_Datasets/GameTox/train.csv"

In [4]:
import pandas as pd

df = pd.read_csv( main_dir.parent / PATH_GAMETOX )

df.head()

,index,message,label
0,30702,no rush,0.0
1,18607,whatever ... watch the replay,0.0
2,32901,useless,1.0
3,25964,3 gunmark,0.0
4,28643,lol,0.0


In [5]:
df["label"] = df["label"].astype(int)
df = df[["message", "label"]]

In [13]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "unitary/toxic-bert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=6,
    ignore_mismatched_sizes=True,
    problem_type="single_label_classification",
    id2label={0: "Non-Toxic", 1: "Insults and Flaming", 2: "Other Offensive Texts", 
              3: "Hate and Harassment 	", 4: "Threats", 5: "Extremism"},
    label2id={"Non-Toxic": 0, "Insults and Flaming": 1, "Other Offensive Texts": 2,
              "Hate and Harassment 	": 3, "Threats": 4, "Extremism": 5}
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4104.56it/s]


In [7]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

dataset

DatasetDict({
    train: Dataset({
        features: ['message', 'label'],
        num_rows: 34367
    })
    test: Dataset({
        features: ['message', 'label'],
        num_rows: 8592
    })
})

In [14]:
def tokenize(batch):

    return tokenizer(
        batch["message"],
        truncation=True,
        padding="max_length",
        max_length=200
    )

In [15]:
tokenized_dataset = dataset.map(
    tokenize,
    batched=True
)

Map: 100%|██████████| 8592/8592 [00:00<00:00, 17583.55 examples/s]


In [10]:
from transformers import TrainingArguments


training_args = TrainingArguments(
    output_dir="./detoxify-GameTox",

    num_train_epochs=3,

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    metric_for_best_model="macro_f1",
    greater_is_better=True, 

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True
)

In [11]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np


def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=1
    )

    return {
        "accuracy":
            accuracy_score(labels, predictions),

        "macro_f1":
            f1_score(
                labels,
                predictions,
                average="macro"
            )
    }

In [16]:
from transformers import Trainer


trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=
        tokenized_dataset["train"],

    eval_dataset=
        tokenized_dataset["test"],

    compute_metrics=
        compute_metrics
)


trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.322874,0.290076,0.903864,0.383449
2,0.243503,0.305922,0.905028,0.442427
3,0.209547,0.346090,0.901420,0.488584


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.66s/it]


TrainOutput(global_step=6444, training_loss=0.2709517345748134, metrics={'train_runtime': 7115.348, 'train_samples_per_second': 14.49, 'train_steps_per_second': 0.906, 'total_flos': 1.05968699877672e+16, 'train_loss': 0.2709517345748134, 'epoch': 3.0})

In [17]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [18]:
def predict_toxicity(text):

    inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
    )

    inputs = {
    key: value.to(device)
    for key, value in inputs.items()
    }


    with torch.no_grad():
        output = model(**inputs)


    probabilities = torch.softmax(
    output.logits,
    dim=1
    )

    pred = torch.argmax(
    probabilities,
    dim=1
    ).item()

    return probabilities, pred

In [25]:
df_test_set = pd.read_csv(main_dir.parent.parent / "WebScraper" / "text_data" / "Steam" / "Counter-Strike_2_labeled.csv")

In [26]:
df_test_set["label"].value_counts()

label
0    452
2     42
1     29
3      2
4      2
Name: count, dtype: int64

In [21]:
model = AutoModelForSequenceClassification.from_pretrained(
    main_dir / "detoxify-GameTox" / "checkpoint-6444",
    id2label={0: "Non-Toxic", 1: "Insults and Flaming", 2: "Other Offensive Texts", 
              3: "Hate and Harassment 	", 4: "Threats", 5: "Extremism"},
    label2id={"Non-Toxic": 0, "Insults and Flaming": 1, "Other Offensive Texts": 2,
              "Hate and Harassment 	": 3, "Threats": 4, "Extremism": 5}
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4568.22it/s]


In [22]:
model.to(device)
model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [27]:
label_list = []

for text in df_test_set["text"]:

    probabilities, pred = predict_toxicity(text)
    label_list.append(model.config.id2label[pred])

In [28]:
label_list = pd.Series(label_list)
label_list.value_counts()

Non-Toxic                410
Insults and Flaming       65
Other Offensive Texts     52
Name: count, dtype: int64